# Loan Status Classification

Algoritma yang akan digunakan:
- Decision Tree Learning (CART)
- Logistic Regression
- Support Vector Machine (SVM)

Library yang digunakan untuk implementasi from scratch adalah numpy. 
Hasil dari ketiga implementasi from scratch tersebut akan dibandingkan dengan hasil algoritma sejenis dari scikit-learn.

## Import Library

In [13]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

print("Import done!")

Import done!


## Load Dataset

In [2]:
train = pd.read_csv('../../src/dtl_lr_svm/data/train.csv')
test = pd.read_csv('../../src/dtl_lr_svm/data/test.csv')

print("Dataset loaded!")

Dataset loaded!


## EDA (Exploratory Data Analysis)

In [3]:
print("=== TRAIN DATASET ===")
print("\nDataset shape:")
print(train.shape)
print("\nFeature types:")
train.info()
print("\nMissing values     :")
print(train.isnull().sum())
print("\nClass distribution :")
print(train["loan_status"].value_counts())
print("\n-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=\n")
print("=== TEST DATASET ===")
print("\nDataset shape:")
print(test.shape)
print("\nFeature types:")
test.info()
print("\nMissing values     :")
print(test.isnull().sum())


=== TRAIN DATASET ===

Dataset shape:
(28800, 13)

Feature types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28800 entries, 0 to 28799
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_id                       28800 non-null  int64  
 1   person_age                      28800 non-null  float64
 2   person_gender                   28800 non-null  object 
 3   person_income                   28800 non-null  float64
 4   person_emp_exp                  28800 non-null  int64  
 5   person_home_ownership           28800 non-null  object 
 6   loan_amnt                       28800 non-null  float64
 7   loan_int_rate                   28800 non-null  float64
 8   loan_percent_income             28800 non-null  float64
 9   cb_person_cred_hist_length      28800 non-null  float64
 10  credit_score                    28800 non-null  int64  
 11  previous_loan_defaults_on_f

## Data Preprocessing

In [4]:
# Remove person_id
train = train.drop(columns=["person_id"])
test = test.drop(columns=["person_id"])

# Separate them into X and y 
# y = f(X)
X = train.drop(columns=["loan_status"])
y = train["loan_status"]

# One-hot encoding
X = pd.get_dummies(X, columns=["person_gender", "person_home_ownership", "previous_loan_defaults_on_file"], drop_first=True)

# Split train-validation
np.random.seed(67)
indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))

train_idx = indices[:split]
val_idx = indices[split:]

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

# Turn into numpy
X_train = X_train.to_numpy(dtype=float)
y_train = y_train.to_numpy(dtype=float)
X_val = X_val.to_numpy(dtype=float)
y_val = y_val.to_numpy(dtype=float)

# Scaling
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

X_train_scaled = (X_train - mean)/std
X_val_scaled = (X_val - mean)/std

print("Preprocessing done!")

Preprocessing done!


## Decision Tree Learning (CART)

### 1. From Scratch

In [5]:
class Node:
    def __init__(self):
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None
        self.prediction = None

def calculateGini(y):
    classes = np.unique(y)
    impurity = 1

    for c in classes:
        pi = np.sum(y==c)/len(y)
        impurity -= pi*pi

    return impurity

def findThresholds(values):
    values = np.unique(values)
    values = np.sort(values)
    numThresholds = 30
    if len(values) <= numThresholds:
        indices = range(len(values)-1)
    else:
        indices = np.linspace(0, len(values)-2, numThresholds, endpoint=True, dtype=int)
    
    thresholds = []

    for i in indices:
        threshold = (values[i]+values[i+1])/2
        thresholds.append(threshold)

    return thresholds

def splitY(X, y, feature, threshold):
    left_y = []
    right_y = []
    for i in range(len(X)):
        if X[i, feature] <= threshold:
            left_y.append(y[i])
        else:
            right_y.append(y[i])
    return left_y, right_y
    
def splitData(X, y, feature, threshold):
    left_y = []
    right_y = []
    left_X = []
    right_X = []
    
    for i in range(len(X)):
        if X[i, feature] <= threshold:
            left_y.append(y[i])
            left_X.append(X[i])
        else:
            right_y.append(y[i])
            right_X.append(X[i])
    return np.array(left_X), np.array(left_y), np.array(right_X), np.array(right_y)
                                                
def weightedGini(left, right):
    total = len(left)+len(right)
    return (len(left)/total)*calculateGini(left) + (len(right)/total)*calculateGini(right)

def findOptimalSplit(X, y):
    bestGini = float("inf")
    bestFeature = None
    bestThreshold = None
    
    for feature in range(X.shape[1]):
        thresholds = findThresholds(X[:, feature])

        for threshold in thresholds:
            left_y, right_y = splitY(X, y, feature, threshold)

            gini = weightedGini(left_y, right_y)
    
            if gini < bestGini:
                bestGini = gini
                bestFeature = feature
                bestThreshold = threshold

    return bestGini, bestFeature, bestThreshold

def majorityClass(y):
    classes, counts = np.unique(y, return_counts=True)
    return classes[np.argmax(counts)]
    
def buildTree(X, y, depth, maxDepth):
    node = Node()

    if depth >= maxDepth:
        node.prediction = majorityClass(y)
        return node
        
    if len(np.unique(y)) == 1:
        node.prediction = y[0]
        return node

    gini, feature, threshold = findOptimalSplit(X, y)

    if feature is None:
        node.prediction = majorityClass(y)
        return node

    node.feature = feature
    node.threshold = threshold

    left_X, left_y, right_X, right_y = splitData(X, y, feature, threshold)

    node.left = buildTree(left_X, left_y, depth+1, maxDepth)
    node.right = buildTree(right_X, right_y, depth+1, maxDepth)

    return node

def predictSample(node, X):
    if node.prediction is not None:
        return node.prediction
        
    if X[node.feature] <= node.threshold:
        return predictSample(node.left, X)
    else:
        return predictSample(node.right, X)
        
def predict(node, X):
    predictions = []

    for x in X:
        prediction = predictSample(node, x)
        predictions.append(prediction)

    return np.array(predictions)
    
tree = buildTree(X_train, y_train, 0, 5)
y_pred_CART_scratch = predict(tree, X_val)

print("=== CART from Scratch ===")
print("F1 Score:", f1_score(y_val, y_pred_CART_scratch, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_CART_scratch))

=== CART from Scratch ===
F1 Score: 0.8619663454483919

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      4461
         1.0       0.86      0.71      0.78      1299

    accuracy                           0.91      5760
   macro avg       0.89      0.84      0.86      5760
weighted avg       0.91      0.91      0.91      5760



### 2. sklearn

In [6]:
model = DecisionTreeClassifier(criterion="gini", max_depth = 5, random_state=67)
model.fit(X_train, y_train)

y_pred_CART_sklearn = model.predict(X_val)

print("=== CART from sklearn ===")
print("F1 Score:", f1_score(y_val, y_pred_CART_sklearn, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_CART_sklearn))

=== CART from sklearn ===
F1 Score: 0.8626686815663194

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      4461
         1.0       0.86      0.72      0.78      1299

    accuracy                           0.91      5760
   macro avg       0.89      0.84      0.86      5760
weighted avg       0.91      0.91      0.91      5760



## Logistic Regression

### 1. From Scratch

In [7]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

def predictProbability(X, weights, bias):
    z = np.dot(X, weights) + bias
    return sigmoid(z)

def train(X, y, learning_rate, epochs):
    weights = np.zeros(X.shape[1])
    bias = 0
    
    for i in range(epochs):
        prediction = predictProbability(X, weights, bias)

        dw = (1/len(y)) * np.dot(X.T, (prediction-y))
        db = (1/len(y)) * np.sum(prediction-y)

        weights -= learning_rate*dw
        bias -= learning_rate*db

    return weights, bias

def predict(X, weights, bias):
    probability = sigmoid(np.dot(X, weights)+bias)

    predictions = np.where(probability >= 0.5, 1, 0)
    return predictions

weights, bias = train(X_train_scaled, y_train, 0.01, 1000)
y_pred_regression_scratch = predict(X_val_scaled, weights, bias)

print("=== Logistic Regression from Scratch ===")
print("F1 Score:", f1_score(y_val, y_pred_regression_scratch, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_regression_scratch))

=== Logistic Regression from Scratch ===
F1 Score: 0.8265081279330442

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.93      0.92      4461
         1.0       0.74      0.72      0.73      1299

    accuracy                           0.88      5760
   macro avg       0.83      0.82      0.83      5760
weighted avg       0.88      0.88      0.88      5760



### 2. sklearn

In [8]:
model = LogisticRegression(random_state=67, max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred_regression_sklearn = model.predict(X_val_scaled)

print("=== Logistic Regression from sklearn ===")
print("F1 Score:", f1_score(y_val, y_pred_regression_sklearn, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_regression_sklearn))

=== Logistic Regression from sklearn ===
F1 Score: 0.8389110055220239

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93      4461
         1.0       0.77      0.73      0.75      1299

    accuracy                           0.89      5760
   macro avg       0.85      0.83      0.84      5760
weighted avg       0.89      0.89      0.89      5760



## Support Vector Machine

### 1. From Scratch

In [12]:
def predict(X, weights, bias):
    scores = np.dot(X, weights) + bias
    predictions = np.where(scores >= 0, 1, -1)
    
    return predictions

def trainSVM(X, y, learning_rate, epochs, lambda_param):
    weights = np.zeros(X.shape[1])
    bias = 0

    for epoch in range(epochs):
        for i in range(len(X)):
            condition = y[i] * (np.dot(X[i], weights)+bias)
            
            if condition >= 1:
                dw = 2 * lambda_param * weights
                db = 0
            else:
                dw = 2 * lambda_param * weights - y[i]*X[i]
                db = -y[i]

            weights -= learning_rate * dw
            bias -= learning_rate * db
    return weights, bias
    
y_train_svm = np.where(y_train == 0, -1, 1)
weights, bias = trainSVM(X_train_scaled, y_train_svm, 0.001, 100, 0.01)
y_pred_SVM_scratch = predict(X_val_scaled, weights, bias)
y_pred_SVM_scratch = np.where(y_pred_SVM_scratch == -1, 0, 1)

print("=== SVM from Scratch ===")
print("F1 Score:", f1_score(y_val, y_pred_SVM_scratch, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_SVM_scratch))

=== SVM from Scratch ===
F1 Score: 0.8387040567921025

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93      4461
         1.0       0.78      0.72      0.75      1299

    accuracy                           0.89      5760
   macro avg       0.85      0.83      0.84      5760
weighted avg       0.89      0.89      0.89      5760



### 2. sklearn

In [14]:
model = LinearSVC(random_state=67, max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred_SVM_sklearn = model.predict(X_val_scaled)

print("=== SVM from sklearn ===")
print("F1 Score:", f1_score(y_val, y_pred_SVM_sklearn, average="macro"))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_SVM_sklearn))

=== SVM from sklearn ===
F1 Score: 0.8395332797600592

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93      4461
         1.0       0.77      0.73      0.75      1299

    accuracy                           0.89      5760
   macro avg       0.85      0.83      0.84      5760
weighted avg       0.89      0.89      0.89      5760



## Model Comparison

In [20]:
results = {
    "Model": [
        "CART Scratch",
        "CART Sklearn",
        "Logistic Regression Scratch",
        "Logistic Regression Sklearn",
        "SVM Scratch",
        "SVM Sklearn"
    ],
    "Macro F1": [
        0.8619,
        0.8627,
        0.8265,
        0.8389,
        0.8387,
        0.8395
    ],
    
    "Accuracy": [
        0.91,
        0.91,
        0.88,
        0.89,
        0.89,
        0.89
    ],

    "Precision": [
        0.89,
        0.89,
        0.83,
        0.85,
        0.85,
        0.85
    ],

    "Recall": [
        0.84,
        0.84,
        0.82,
        0.83,
        0.83,
        0.83
    ]
}

df_results = pd.DataFrame(results)
df_results.sort_values(
    by="Macro F1",
    ascending=False
)

,Model,Macro F1,Accuracy,Precision,Recall
1,CART Sklearn,0.8627,0.91,0.89,0.84
0,CART Scratch,0.8619,0.91,0.89,0.84
5,SVM Sklearn,0.8395,0.89,0.85,0.83
3,Logistic Regression Sklearn,0.8389,0.89,0.85,0.83
4,SVM Scratch,0.8387,0.89,0.85,0.83
2,Logistic Regression Scratch,0.8265,0.88,0.83,0.82


Berdasarkan hasil Macro F1, CART dengan implementasi sklearn memiliki nilai tertinggi. Nilai kedua tertinggi adalah CART yang dibuat dari scratch. Hal ini terjadi karena dataset kemungkinan memiliki pola non-linear, ditangkap baik oleh decision tree. 

Logistic Regression dan SVM memperoleh nilai yang lebih rendah. Model ini mengasumsikan decision boundary yang linear. Akibatnya, mereka cukup terbatas dalam merepresentasikan hubungan yang kompleks antara features.

Implementasi sklearn sedikit lebih baik dibandingkan implementasi dari scratch. Terdapat perbedaan kecil di SVM dan CART. Sementara itu, terdapat perbedaan yang sedikit lebih besar pada Logistic Regression karena perbedaan metode optimasi dan metode regulasi.